# C5-00b · TITANIC — ETL, Feature Engineering y Preprocessing
### Capítulo 5 · Machine Learning 1: Aprendizaje supervisado · Samsung Innovation Campus
**Puente Probabilidad & Estadística → Machine Learning**

**Tipo de caso:** clasificación **binaria tabular** con datos **mixtos y desprolijos**: el *caso del mundo real*.
Numéricas, conteos, una ordinal, nominales, nulos con distintos mecanismos y una columna que filtra el target.

**Objetivos**
1. Hacer un ETL que detecte y elimine **leakage** y columnas duplicadas.
2. Diagnosticar el **mecanismo** de cada faltante (MCAR / MAR / MNAR).
3. Crear features justificadas: familia, solo, log_fare, tiene_cabina.
4. Imputar la edad **por grupo** (MAR) con medianas aprendidas solo en train.
5. Preparar tres tareas: clasificación, regresión y **Poisson**.
6. Exportar el módulo `cap5_titanic.py`.

**Cómo encaja este Colab en el capítulo**

```
CAPA BASE (este Colab)                      CAPA MODELO
ETL -> Feature Engineering -> Preprocessing -> módulo cap5_titanic.py -> C5-M01 ... C5-M12
```

Al final se exporta el módulo **`cap5_titanic.py`** con la **misma interfaz** que los otros dos datasets:

| Función | Devuelve |
|---|---|
| `cargar(tarea)` | `X` (DataFrame) e `y` (Series) listos para modelar |
| `preprocesador(tarea, ...)` | un `Pipeline` / `ColumnTransformer` **sin ajustar** |
| `dividir(X, y, tarea)` | train/test (estratificado si es clasificación) |
| `INFO` | ficha del dataset: tipo de caso, tareas, métrica sugerida |

Los Colabs de modelos solo cambian `DATASET = "titanic"` y reutilizan todo esto.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

SEED = 42
pd.set_option("display.precision", 3)

## 2. ETL

| Etapa | Qué hacemos | Puente P&E |
|---|---|---|
| **E**xtract | `sns.load_dataset("titanic")` (o el CSV de seaborn-data) | muestra de n = 891 pasajeros |
| **T**ransform | quitar leakage y duplicadas, fijar tipos | qué variables aleatorias son legítimas |
| **L**oad | CSV limpio | muestra fija y auditable |

In [ ]:
# Extract
crudo = sns.load_dataset("titanic")
print("Forma cruda:", crudo.shape)
crudo.head()

In [ ]:
# ¿Qué columnas son redundantes o filtran el target?
chequeos = pd.DataFrame({
    "alive == survived": [(crudo["alive"].map({"yes": 1, "no": 0}) == crudo["survived"]).mean()],
    "class <-> pclass": [pd.crosstab(crudo["class"], crudo["pclass"]).gt(0).sum(axis=1).eq(1).all()],
    "embark_town <-> embarked": [pd.crosstab(crudo["embark_town"], crudo["embarked"]).gt(0).sum(axis=1).eq(1).all()],
}).T.rename(columns={0: "resultado"})
chequeos

> **Puente P&E — leakage:** `alive` coincide con `survived` en el 100% de las filas: es el **target
> con otro nombre**. Con ella, P(Y | X) sería trivial en train e imposible en producción.
> `class` y `embark_town` son **la misma variable aleatoria** codificada de otra forma.

In [ ]:
# Transform
df = crudo.drop(columns=["alive", "class", "embark_town", "who", "adult_male", "alone"]).copy()
df["sex"] = df["sex"].astype(str)
df["embarked"] = df["embarked"].astype(object)

ficha = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "%_nulos": 100 * df.isna().mean(),
    "valores_unicos": df.nunique(),
})
print("Filas duplicadas:", df.duplicated().sum())
ficha

> **¿107 duplicados = error?** No. Esta versión del dataset **no tiene nombre ni ticket**: dos pasajeros
> de 3ra clase, hombres, sin edad, con la misma tarifa y el mismo puerto son **personas distintas** con
> la misma descripción. Borrarlos sesgaría la muestra (sobre todo en 3ra clase).

In [ ]:
# Load
df.to_csv("titanic_limpio.csv", index=False)

## 3. EDA orientado a decisiones

### 3.1 El target y su prior

In [ ]:
prior = df["survived"].value_counts(normalize=True).rename("P(Y)")
prior.index = ["no sobrevive (0)", "sobrevive (1)"]
prior.to_frame()

> **Puente P&E:** prior P(sobrevive) ≈ 0.38. Moderadamente desbalanceado: el baseline "nadie sobrevive"
> acierta ~62%. Conviene **estratificar** el split para que train y test mantengan esa proporción.

### 3.2 Probabilidades condicionales por grupo

In [ ]:
tablas = []
for c in ["sex", "pclass", "embarked"]:
    t = df.groupby(c)["survived"].agg(["mean", "size"]).rename(columns={"mean": "P(sobrevive | grupo)", "size": "n"})
    t.index = [f"{c} = {i}" for i in t.index]
    tablas.append(t)
pd.concat(tablas)

> **Puente P&E:** cada fila es un **estimador** de P(Y = 1 | grupo). Con n chico (ej. embarked = Q)
> el estimador tiene más varianza: ojo con sacar conclusiones fuertes de grupos pequeños.

### 3.3 Mecanismo de los faltantes

In [ ]:
mecanismo = []
for c in ["age", "deck", "embarked"]:
    falta = df[c].isna()
    if falta.sum() < 5:
        mecanismo.append({"columna": c, "n_faltantes": int(falta.sum()),
                          "P(sobrevive | falta)": np.nan, "P(sobrevive | presente)": np.nan,
                          "p_valor_chi2": np.nan})
        continue
    tabla = pd.crosstab(falta, df["survived"])
    chi2, p, *_ = stats.chi2_contingency(tabla)
    mecanismo.append({
        "columna": c, "n_faltantes": int(falta.sum()),
        "P(sobrevive | falta)": df.loc[falta, "survived"].mean(),
        "P(sobrevive | presente)": df.loc[~falta, "survived"].mean(),
        "p_valor_chi2": p,
    })
pd.DataFrame(mecanismo).set_index("columna")

In [ ]:
# ¿El faltante de edad depende de otras variables observadas? (MAR)
pd.crosstab(df["pclass"], df["age"].isna(), normalize="index").rename(
    columns={False: "edad presente", True: "edad faltante"})

> **Puente P&E — tres mecanismos, tres decisiones:**
>
> | Columna | Hallazgo | Mecanismo probable | Decisión |
> |---|---|---|---|
> | `deck` (77% nulos) | sobreviven 67% con cabina vs. 30% sin | **MNAR / informativo** | eliminar `deck` y crear flag `tiene_cabina` |
> | `age` (20% nulos) | faltan mucho más en 3ra clase | **MAR** (depende de pclass) | imputar con la mediana de **pclass × sex** + flag `age_faltante` |
> | `embarked` (2 nulos) | casos aislados | ~**MCAR** | imputar la moda |
>
> El test chi-cuadrado contrasta H0: "faltar es independiente de sobrevivir".

In [ ]:
# 3.4 Asimetría de fare y conteos
resumen = pd.DataFrame({
    "media": df[["fare", "sibsp", "parch", "age"]].mean(),
    "varianza": df[["fare", "sibsp", "parch", "age"]].var(),
    "asimetria": df[["fare", "sibsp", "parch", "age"]].skew(),
})
resumen.loc["log1p(fare)"] = [np.log1p(df["fare"]).mean(), np.log1p(df["fare"]).var(), np.log1p(df["fare"]).skew()]
print("Tarifas = 0:", (df["fare"] == 0).sum(), "-> por eso log1p y no log")
resumen

> **Puente P&E:** `fare` tiene asimetría fuerte (cola larga de 1ra clase); `log1p` la lleva cerca de 0.
> `sibsp` y `parch` son **conteos** con varianza mayor que la media: señal de **sobredispersión**
> (relevante para la regresión de Poisson del C5-M03).

## 4. Feature Engineering

| Hallazgo | Feature | Tipo | Justificación P&E |
|---|---|---|---|
| sibsp y parch miden lo mismo (acompañantes) | `familia = sibsp + parch` | conteo | suma de conteos; target de la tarea Poisson |
| viajar solo podría cambiar la supervivencia | `solo = (familia == 0)` | binaria | indicador de un evento |
| fare muy asimétrica, con ceros | `log_fare = log(1 + fare)` | continua | estabiliza varianza, acerca a simetría |
| deck faltante es informativo | `tiene_cabina = deck no es nulo` | binaria | el **nulo como señal** (MNAR) |

Todas son **fila a fila** ⇒ sin leakage. La imputación de edad **sí aprende** de los datos ⇒ va en el pipeline.

In [ ]:
df_fe = df.copy()
df_fe["familia"] = df_fe["sibsp"] + df_fe["parch"]
df_fe["solo"] = (df_fe["familia"] == 0).astype(int)
df_fe["log_fare"] = np.log1p(df_fe["fare"])
df_fe["tiene_cabina"] = df_fe["deck"].notna().astype(int)
df_fe = df_fe.drop(columns=["deck", "fare"])

# Validación rápida de cada feature nueva: P(sobrevive | feature)
pd.concat([
    df_fe.groupby("solo")["survived"].mean().rename(lambda i: f"solo = {i}"),
    df_fe.groupby("tiene_cabina")["survived"].mean().rename(lambda i: f"tiene_cabina = {i}"),
    df_fe.groupby(pd.cut(df_fe["familia"], [-1, 0, 3, 10]), observed=True)["survived"].mean()
         .rename(lambda i: f"familia en {i}"),
]).rename("P(sobrevive)").to_frame()

> **Puente P&E:** la relación de `familia` con sobrevivir **no es monótona** (familias medianas
> sobreviven más que solos y que familias grandes). Un modelo lineal necesitará ayuda (binning o
> término cuadrático); un árbol la captura solo.

## 5. Imputación por grupo (MAR) como transformador de sklearn

Como la imputación **aprende medianas**, tiene que ser un transformador que se ajusta **solo en train**.

In [ ]:
class ImputadorPorGrupo(BaseEstimator, TransformerMixin):
    """Imputa una columna con la mediana de su grupo (ej. pclass x sex)
    y agrega un flag <columna>_faltante. Las medianas salen de train."""

    def __init__(self, columna="age", grupos=("pclass", "sex")):
        self.columna = columna
        self.grupos = grupos

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.medianas_ = X.groupby(list(self.grupos))[self.columna].median()
        self.mediana_global_ = float(X[self.columna].median())
        self.columnas_entrada_ = list(X.columns)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columnas_entrada_).copy()
        faltante = X[self.columna].isna().to_numpy()
        claves = list(zip(*[X[g] for g in self.grupos]))
        valores = np.array([self.medianas_.get(k, np.nan) for k in claves],
                           dtype=float)
        valores = np.where(np.isnan(valores), self.mediana_global_, valores)
        X[self.columna] = np.where(faltante, valores, X[self.columna])
        X[self.columna + "_faltante"] = faltante.astype(int)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.array(self.columnas_entrada_ + [self.columna + "_faltante"])

# Demostración: medianas aprendidas solo con train
X_demo = df_fe.drop(columns=["survived"])
y_demo = df_fe["survived"]
Xtr_d, Xte_d, _, _ = train_test_split(X_demo, y_demo, test_size=0.2, random_state=SEED, stratify=y_demo)
imp = ImputadorPorGrupo().fit(Xtr_d)
imp.medianas_.rename("mediana de edad (train)").to_frame()

> **Puente P&E:** imputar con la mediana **global** ignora que los de 1ra clase son mayores (MAR):
> sesga la edad de 3ra hacia arriba. La mediana **por grupo** es un estimador de E[age | pclass, sex]
> más robusto ante outliers. El flag `age_faltante` conserva la información de que el dato faltaba.

## 6. Preprocessing

| Grupo | Columnas | Transformación |
|---|---|---|
| Edad | `age` | `ImputadorPorGrupo` → + flag `age_faltante` |
| Numéricas | `age, sibsp, parch, familia, log_fare, age_faltante` | `StandardScaler` (opcional) |
| Ordinal | `pclass` (1 < 2 < 3) | se mantiene como número, escalada (opcional) |
| Binarias | `solo, tiene_cabina` | passthrough |
| Nominales | `sex, embarked` | moda + `OneHotEncoder` (`drop="if_binary"`) |

In [ ]:
numericas = ["age", "sibsp", "parch", "familia", "log_fare", "age_faltante"]
ordinal = ["pclass"]
binarias = ["solo", "tiene_cabina"]
categoricas = ["sex", "embarked"]

ct = ColumnTransformer([
    ("num", StandardScaler(), numericas),
    ("ord", StandardScaler(), ordinal),
    ("bin", "passthrough", binarias),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(drop="if_binary", handle_unknown="ignore",
                                            sparse_output=False))]), categoricas),
], verbose_feature_names_out=False)
prep = Pipeline([("imputar_edad", ImputadorPorGrupo()), ("columnas", ct)])

X = df_fe.drop(columns=["survived"])
y = df_fe["survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

prep.fit(X_train)   # <- solo train
Xt_train = pd.DataFrame(prep.transform(X_train), columns=prep.get_feature_names_out())
print("Features finales:", list(Xt_train.columns))
print("Nulos tras preprocesar:", int(Xt_train.isna().sum().sum()))
Xt_train.describe().T[["mean", "std", "min", "max"]]

## 7. Verificación: baseline vs. dos modelos simples

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
modelos = {
    "Dummy (prior)": DummyClassifier(strategy="most_frequent"),
    "Logística": LogisticRegression(max_iter=1000),
    "Árbol (prof. 4)": DecisionTreeClassifier(max_depth=4, random_state=SEED),
}
res = {}
for nombre, m in modelos.items():
    s = cross_val_score(Pipeline([("prep", prep), ("modelo", m)]),
                        X_train, y_train, cv=cv, scoring="accuracy")
    res[nombre] = {"accuracy_media": s.mean(), "desvio": s.std()}
pd.DataFrame(res).T

> **Puente P&E:** la diferencia entre modelos hay que leerla junto al **desvío entre folds**.
> Si dos modelos difieren menos que su desvío, todavía no hay evidencia de que uno sea mejor (C5-10).

## 8. Tres tareas desde el mismo dataset

| Tarea | Target | Se excluye de X | Por qué |
|---|---|---|---|
| `clasificacion` | `survived` | — | — |
| `regresion` | `log_fare` | `log_fare` | es el target |
| `poisson` | `familia` | `sibsp, parch, familia, solo` | se calculan **con** el target ⇒ leakage |

In [ ]:
%%writefile cap5_titanic.py
"""cap5_titanic.py - ETL + FE + Preprocessing de TITANIC (Capítulo 5, SIC).
Interfaz común: cargar(tarea), preprocesador(tarea, ...), dividir(...), INFO."""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

SEED = 42
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

INFO = {
    "nombre": "titanic",
    "tipo_caso": "clasificación binaria tabular con datos mixtos y nulos (mundo real)",
    "tareas": {
        "clasificacion": "y = survived (1 sobrevive)",
        "regresion": "y = log_fare = log(1 + fare)",
        "poisson": "y = familia = sibsp + parch (conteo)",
    },
    "metrica_sugerida": {"clasificacion": "accuracy / ROC-AUC",
                         "regresion": "RMSE", "poisson": "deviance de Poisson"},
}

class ImputadorPorGrupo(BaseEstimator, TransformerMixin):
    """Imputa una columna con la mediana de su grupo (ej. pclass x sex)
    y agrega un flag <columna>_faltante. Las medianas salen de train."""

    def __init__(self, columna="age", grupos=("pclass", "sex")):
        self.columna = columna
        self.grupos = grupos

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.medianas_ = X.groupby(list(self.grupos))[self.columna].median()
        self.mediana_global_ = float(X[self.columna].median())
        self.columnas_entrada_ = list(X.columns)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columnas_entrada_).copy()
        faltante = X[self.columna].isna().to_numpy()
        claves = list(zip(*[X[g] for g in self.grupos]))
        valores = np.array([self.medianas_.get(k, np.nan) for k in claves],
                           dtype=float)
        valores = np.where(np.isnan(valores), self.mediana_global_, valores)
        X[self.columna] = np.where(faltante, valores, X[self.columna])
        X[self.columna + "_faltante"] = faltante.astype(int)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.array(self.columnas_entrada_ + [self.columna + "_faltante"])


def _etl():
    try:
        import seaborn as sns
        df = sns.load_dataset("titanic")
    except Exception:
        df = pd.read_csv(URL)
    # alive = target repetido (leakage); class y embark_town duplican pclass y embarked;
    # who / adult_male / alone son derivadas que recalculamos o no usamos.
    df = df.drop(columns=["alive", "class", "embark_town", "who", "adult_male", "alone"])
    df["sex"] = df["sex"].astype(str)
    df["embarked"] = df["embarked"].astype(object)
    return df


def _feature_engineering(df):
    df = df.copy()
    df["familia"] = df["sibsp"] + df["parch"]
    df["solo"] = (df["familia"] == 0).astype(int)
    df["log_fare"] = np.log1p(df["fare"])
    df["tiene_cabina"] = df["deck"].notna().astype(int)   # nulo MNAR -> flag
    return df.drop(columns=["deck", "fare"])


COLUMNAS = {
    "clasificacion": ["pclass", "sex", "age", "sibsp", "parch", "familia",
                      "solo", "log_fare", "embarked", "tiene_cabina"],
    "regresion": ["pclass", "sex", "age", "sibsp", "parch", "familia",
                  "solo", "embarked", "tiene_cabina"],
    "poisson": ["pclass", "sex", "age", "log_fare", "embarked", "tiene_cabina"],
}
TARGET = {"clasificacion": "survived", "regresion": "log_fare", "poisson": "familia"}


def cargar(tarea="clasificacion"):
    if tarea not in TARGET:
        raise ValueError(f"tarea debe ser una de {list(TARGET)}")
    df = _feature_engineering(_etl())
    return df[COLUMNAS[tarea]].copy(), df[TARGET[tarea]].copy()


def preprocesador(tarea="clasificacion", escalar=True):
    cols = COLUMNAS[tarea]
    numericas = [c for c in ["age", "sibsp", "parch", "familia", "log_fare"] if c in cols]
    numericas.append("age_faltante")
    ordinal = ["pclass"]
    binarias = [c for c in ["solo", "tiene_cabina"] if c in cols]
    categoricas = ["sex", "embarked"]
    escalador = StandardScaler() if escalar else "passthrough"
    ct = ColumnTransformer([
        ("num", escalador, numericas),
        ("ord", escalador, ordinal),
        ("bin", "passthrough", binarias),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(drop="if_binary", handle_unknown="ignore",
                                  sparse_output=False)),
        ]), categoricas),
    ], verbose_feature_names_out=False)
    return Pipeline([("imputar_edad", ImputadorPorGrupo("age", ("pclass", "sex"))),
                     ("columnas", ct)])


def dividir(X, y, tarea="clasificacion", test_size=0.2):
    estratos = y if tarea == "clasificacion" else None
    return train_test_split(X, y, test_size=test_size,
                            random_state=SEED, stratify=estratos)

In [ ]:
import importlib, cap5_titanic
importlib.reload(cap5_titanic)
from cap5_titanic import cargar, preprocesador, dividir, INFO

filas = []
for tarea in ["clasificacion", "regresion", "poisson"]:
    X_, y_ = cargar(tarea)
    Xtr, Xte, ytr, yte = dividir(X_, y_, tarea)
    p = preprocesador(tarea).fit(Xtr)
    filas.append({"tarea": tarea, "target": y_.name, "n": len(X_),
                  "features_entrada": X_.shape[1],
                  "features_salida": len(p.get_feature_names_out()),
                  "media_y": y_.mean(), "var_y": y_.var()})
pd.DataFrame(filas).set_index("tarea")

> **Puente P&E — tarea Poisson:** compará `media_y` y `var_y` de `familia`. En una Poisson,
> media = varianza; acá la varianza es casi el triple ⇒ **sobredispersión**: en el C5-M03 conviene
> comparar con una **Binomial negativa**.

In [ ]:
# (Opcional) Guardar el módulo en Google Drive para reutilizarlo en los Colabs de modelos.
USAR_DRIVE = False

if USAR_DRIVE:
    from google.colab import drive
    import shutil, os
    drive.mount("/content/drive")
    destino = "/content/drive/MyDrive/SIC_cap5"
    os.makedirs(destino, exist_ok=True)
    shutil.copy("cap5_titanic.py", destino)
    print("Módulo guardado en", destino)
    # En los Colabs de modelos:
    #   import sys; sys.path.append("/content/drive/MyDrive/SIC_cap5")
    #   from cap5_titanic import cargar, preprocesador, dividir, INFO

## 9. Ejercicio propuesto

1. Reemplazá `ImputadorPorGrupo` por `SimpleImputer(strategy="median")` (mediana global) y compará la
   accuracy de la logística. ¿La imputación MAR mejora?
2. Agregá una feature `familia_cat` con los cortes {0}, {1-3}, {4+} y verificá si la logística mejora
   (hallazgo de la sección 4: relación no monótona).
3. Cargá la versión de OpenML (`fetch_openml("titanic", version=1, as_frame=True)`), extraé el **título**
   del nombre (Mr, Mrs, Miss, Master) e imputá la edad por título. ¿Qué columnas de OpenML filtran el target?
   (pista: `boat`, `body`).
4. Repetí el test chi-cuadrado de la sección 3.3 para `age` faltante vs. `pclass`. ¿Qué concluís?

## 10. Cierre: tabla puente

| Concepto P&E | Herramienta ML en este Colab |
|---|---|
| Leakage = variable que es el target | eliminar `alive` |
| Misma variable aleatoria duplicada | eliminar `class`, `embark_town` |
| Prior P(Y) y estratificación | `value_counts`, `stratify=y` |
| P(Y \| grupo) como estimador | `groupby(...)["survived"].mean()` |
| MCAR / MAR / MNAR | moda / mediana por grupo / flag `tiene_cabina` |
| Test de independencia | chi-cuadrado faltante vs. target |
| Asimetría y transformación | `log1p(fare)` |
| Conteos y sobredispersión | `familia`: varianza vs. media |
| Estimadores ajustados solo en train | `ImputadorPorGrupo`, `StandardScaler` en `Pipeline` |

## 11. Autoevaluación
1. ¿Por qué `alive` es leakage y `tiene_cabina` no?
2. ¿Qué evidencia indica que el faltante de `age` es MAR?
3. ¿Por qué no se eliminan los 107 duplicados?
4. ¿Por qué en la tarea Poisson se excluyen `sibsp` y `parch`?